# Load Dataset CIFAR-10

In [2]:
import tensorflow as tf
import numpy as np

# 1. Load dataset CIFAR-10 langsung dari internet lewat Keras
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.cifar10.load_data()

# 2. Saring agar hanya mengambil 2 kelas saja: Kucing (label 3) dan Anjing (label 5)
# Kamu bebas ganti kelas lain, tapi ini biar pas dengan tema tugasmu
keep_train = np.where((y_train_full == 3) | (y_train_full == 5))[0]
keep_test = np.where((y_test_full == 3) | (y_test_full == 5))[0]

x_train_filtered = x_train_full[keep_train]
y_train_filtered = y_train_full[keep_train]
x_test_filtered = x_test_full[keep_test]
y_test_filtered = y_test_full[keep_test]

# Ubah label menjadi biner: Kucing = 0, Anjing = 1
y_train_filtered = np.where(y_train_filtered == 3, 0, 1).astype('float32')
y_test_filtered = np.where(y_test_filtered == 3, 0, 1).astype('float32')

# 3. Bagi data test menjadi Validation (15%) dan Testing (15%) sesuai aturan tugas
from sklearn.model_selection import train_test_split
x_val, x_test, y_val, y_test = train_test_split(
    x_test_filtered, y_test_filtered, test_size=0.5, random_state=42
)

print(f"Data Training   : {x_train_filtered.shape[0]} gambar")
print(f"Data Validation : {x_val.shape[0]} gambar")
print(f"Data Testing    : {x_test.shape[0]} gambar")
print("✅ Dataset CIFAR-10 2 Kelas sukses dimuat dan bebas dari data rusak!")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1525s 9us/step
Data Training   : 10000 gambar
Data Validation : 1000 gambar
Data Testing    : 1000 gambar
✅ Dataset CIFAR-10 2 Kelas sukses dimuat dan bebas dari data rusak!


# Arsitektur CNN

In [3]:
from tensorflow.keras import layers, models

model_scratch = models.Sequential([
    # Input disesuaikan dengan ukuran CIFAR-10 yaitu 32x32 piksel
    layers.Input(shape=(32, 32, 3)),
    layers.Rescaling(1./255),
    
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Flatten & Classifier
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    
    # Output layer (Binary Classification)
    layers.Dense(1, activation='sigmoid')
])

model_scratch.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_scratch.summary()

I0000 00:00:1781272204.704308      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 544,833 (2.08 MB)

 Trainable params: 544,385 (2.08 MB)

 Non-trainable params: 448 (1.75 KB)

# Training

In [4]:
EPOCHS = 20
BATCH_SIZE = 32

history_scratch = model_scratch.fit(
    x_train_filtered, y_train_filtered,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

Epoch 1/20


I0000 00:00:1781272214.489357     137 service.cc:152] XLA service 0x7a59000121d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781272214.489393     137 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1781272215.027921     137 cuda_dnn.cc:529] Loaded cuDNN version 91002


 51/313 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5507 - loss: 0.9886

I0000 00:00:1781272218.886490     137 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.6094 - loss: 0.7495 - val_accuracy: 0.5440 - val_loss: 1.0168
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6639 - loss: 0.6212 - val_accuracy: 0.6550 - val_loss: 0.6189
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7021 - loss: 0.5747 - val_accuracy: 0.6630 - val_loss: 0.6039
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7179 - loss: 0.5481 - val_accuracy: 0.7010 - val_loss: 0.5662
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7382 - loss: 0.5262 - val_accuracy: 0.6720 - val_loss: 0.5850
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7393 - loss: 0.5217 - val_accuracy: 0.6770 - val_loss: 0.6342
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7539 - loss: 0.5035 - val_accuracy: 0.6930 - val_loss: 0.6714
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7612 - loss: 0.4887 - val_accuracy: 0.7160 - va

# Arsitektur Transfer

In [16]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Load Pretrained Model MobileNetV2 (Kembali ke kode awal yang aman)
base_model_cifar = tf.keras.applications.MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False,
    weights='imagenet'
)

# 2. Bekukan seluruh layer pretrained (Feature Extraction)
base_model_cifar.trainable = False

# 3. Bangun Top Classifier Baru
model_tl_cifar = models.Sequential([
    base_model_cifar,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid') # Biner: Kucing vs Anjing
])

# 4. Compile Model
model_tl_cifar.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✅ Model Transfer Learning MobileNetV2 untuk CIFAR-10 siap!")

/tmp/ipykernel_58/1421604359.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model_cifar = tf.keras.applications.MobileNetV2(


✅ Model Transfer Learning MobileNetV2 untuk CIFAR-10 siap!


# Training Transfer Learning

In [17]:
EPOCHS_TL = 10
BATCH_SIZE = 32

history_tl = model_tl_cifar.fit(
    x_train_filtered, y_train_filtered,
    validation_data=(x_val, y_val),
    epochs=EPOCHS_TL,
    batch_size=BATCH_SIZE
)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - accuracy: 0.5191 - loss: 0.6911 - val_accuracy: 0.5420 - val_loss: 0.6886
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5357 - loss: 0.6885 - val_accuracy: 0.5440 - val_loss: 0.6869
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5379 - loss: 0.6877 - val_accuracy: 0.5440 - val_loss: 0.6859
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5399 - loss: 0.6868 - val_accuracy: 0.5510 - val_loss: 0.6849
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5428 - loss: 0.6858 - val_accuracy: 0.5470 - val_loss: 0.6848
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5411 - loss: 0.6857 - val_accuracy: 0.5510 - val_loss: 0.6842
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5426 - loss: 0.6846 - val_accuracy: 0.5500 - val_loss: 0.6836
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.5415 - loss: 0.6845 - val_accuracy: 